In [23]:
import json

with open("data/raw/1535465.json", "r", encoding="utf-8") as f:
    match = json.load(f)

print("Match loaded successfully!")

Match loaded successfully!


In [24]:
type(match)
match.keys()
match["info"].keys()
len(match["innings"])

2

In [25]:
# Basic match information

info = match["info"]

print("Date:", info["dates"])
print("Season:", info["season"])
print("Venue:", info["venue"])
print("Teams:", info["teams"])
print("Toss:", info["toss"])
print("Outcome:", info["outcome"])
print("Player of the Match:", info["player_of_match"])

Date: ['2026-05-31']
Season: 2026
Venue: Narendra Modi Stadium, Ahmedabad
Teams: ['Gujarat Titans', 'Royal Challengers Bengaluru']
Toss: {'decision': 'field', 'winner': 'Royal Challengers Bengaluru'}
Outcome: {'winner': 'Royal Challengers Bengaluru', 'by': {'wickets': 5}}
Player of the Match: ['V Kohli']


In [26]:
# Explore innings structure

print("Number of innings:", len(match["innings"]))

for i, innings in enumerate(match["innings"], start=1):
    print(f"Innings {i}: {innings['team']}")

Number of innings: 2
Innings 1: Gujarat Titans
Innings 2: Royal Challengers Bengaluru


In [27]:
# Explore first innings

first_innings = match["innings"][0]

print("Batting team:", first_innings["team"])
print("Keys:", first_innings.keys())
print("Number of overs:", len(first_innings["overs"]))
print("Powerplays:", first_innings["powerplays"])

Batting team: Gujarat Titans
Keys: dict_keys(['team', 'overs', 'powerplays'])
Number of overs: 20
Powerplays: [{'from': 0.1, 'to': 5.6, 'type': 'mandatory'}]


In [28]:
# Explore first over

first_over = first_innings["overs"][0]

print("Over number:", first_over["over"])
print("Keys:", first_over.keys())
print("Number of delivery records:", len(first_over["deliveries"]))

Over number: 0
Keys: dict_keys(['over', 'deliveries'])
Number of delivery records: 9


In [29]:
# Inspect every delivery in the first over

deliveries = first_over["deliveries"]

for delivery in deliveries:
    print(
        delivery["actual_delivery"],
        "| Batter:", delivery["batter"],
        "| Bowler:", delivery["bowler"],
        "| Runs:", delivery["runs"]
    )

0.1 | Batter: B Sai Sudharsan | Bowler: JA Duffy | Runs: {'batter': 0, 'extras': 0, 'total': 0}
0.2 | Batter: B Sai Sudharsan | Bowler: JA Duffy | Runs: {'batter': 0, 'extras': 0, 'total': 0}
0.3 | Batter: B Sai Sudharsan | Bowler: JA Duffy | Runs: {'batter': 4, 'extras': 0, 'total': 4}
0.4 | Batter: B Sai Sudharsan | Bowler: JA Duffy | Runs: {'batter': 4, 'extras': 0, 'total': 4}
0.5 | Batter: B Sai Sudharsan | Bowler: JA Duffy | Runs: {'batter': 0, 'extras': 1, 'total': 1}
0.5 | Batter: B Sai Sudharsan | Bowler: JA Duffy | Runs: {'batter': 0, 'extras': 1, 'total': 1}
0.5 | Batter: B Sai Sudharsan | Bowler: JA Duffy | Runs: {'batter': 1, 'extras': 0, 'total': 1}
0.6 | Batter: Shubman Gill | Bowler: JA Duffy | Runs: {'batter': 0, 'extras': 1, 'total': 1}
0.6 | Batter: Shubman Gill | Bowler: JA Duffy | Runs: {'batter': 1, 'extras': 0, 'total': 1}


In [30]:
# Inspect deliveries containing extras

for delivery in deliveries:
    if "extras" in delivery:
        print(
            delivery["actual_delivery"],
            "| Extras:", delivery["extras"],
            "| Runs:", delivery["runs"]
        )

0.5 | Extras: {'wides': 1} | Runs: {'batter': 0, 'extras': 1, 'total': 1}
0.5 | Extras: {'wides': 1} | Runs: {'batter': 0, 'extras': 1, 'total': 1}
0.6 | Extras: {'wides': 1} | Runs: {'batter': 0, 'extras': 1, 'total': 1}


In [31]:
# Identify wide deliveries

for delivery in deliveries:
    is_wide = (
        "extras" in delivery
        and "wides" in delivery["extras"]
    )

    print(
        delivery["actual_delivery"],
        "| is_wide:", is_wide
    )

0.1 | is_wide: False
0.2 | is_wide: False
0.3 | is_wide: False
0.4 | is_wide: False
0.5 | is_wide: True
0.5 | is_wide: True
0.5 | is_wide: False
0.6 | is_wide: True
0.6 | is_wide: False


In [32]:
# Explore different types of extras in the first innings

extra_counts = {}

for over in first_innings["overs"]:
    for delivery in over["deliveries"]:
        if "extras" in delivery:
            for extra_type, value in delivery["extras"].items():
                extra_counts[extra_type] = extra_counts.get(extra_type, 0) + value

print("Extra types found:")
for extra_type, count in extra_counts.items():
    print(f"{extra_type}: {count}")

Extra types found:
wides: 4
legbyes: 1


In [33]:
# Find different types of delivery records

delivery_types = set()

for over in first_innings["overs"]:
    for delivery in over["deliveries"]:
        if "extras" in delivery:
            delivery_types.add(tuple(sorted(delivery["extras"].keys())))
        else:
            delivery_types.add(("none",))

print("Different delivery-extra combinations:")

for delivery_type in sorted(delivery_types):
    print(delivery_type)

Different delivery-extra combinations:
('legbyes',)
('none',)
('wides',)


In [34]:
# Inspect no-ball deliveries

no_ball_count = 0

for over in first_innings["overs"]:
    for delivery in over["deliveries"]:
        if "extras" in delivery and "noballs" in delivery["extras"]:
            no_ball_count += 1
            print(delivery)

print("Total no-ball delivery records:", no_ball_count)

Total no-ball delivery records: 0


In [35]:
# Test a basic legal-delivery rule

total_delivery_records = 0
legal_deliveries = 0
wide_deliveries = 0
no_ball_deliveries = 0

for over in first_innings["overs"]:
    for delivery in over["deliveries"]:
        total_delivery_records += 1

        extras = delivery.get("extras", {})

        is_wide = "wides" in extras
        is_no_ball = "noballs" in extras

        if is_wide:
            wide_deliveries += 1

        if is_no_ball:
            no_ball_deliveries += 1

        if not is_wide and not is_no_ball:
            legal_deliveries += 1

print("Total delivery records:", total_delivery_records)
print("Wide deliveries:", wide_deliveries)
print("No-ball deliveries:", no_ball_deliveries)
print("Legal deliveries:", legal_deliveries)

Total delivery records: 124
Wide deliveries: 4
No-ball deliveries: 0
Legal deliveries: 120


In [36]:
# Check legal balls in each over

for over in first_innings["overs"]:
    legal_balls = 0

    for delivery in over["deliveries"]:
        extras = delivery.get("extras", {})

        is_wide = "wides" in extras
        is_no_ball = "noballs" in extras

        if not is_wide and not is_no_ball:
            legal_balls += 1

    print(
        f"Over {over['over']}: "
        f"{len(over['deliveries'])} delivery records, "
        f"{legal_balls} legal balls"
    )

Over 0: 9 delivery records, 6 legal balls
Over 1: 6 delivery records, 6 legal balls
Over 2: 6 delivery records, 6 legal balls
Over 3: 6 delivery records, 6 legal balls
Over 4: 6 delivery records, 6 legal balls
Over 5: 6 delivery records, 6 legal balls
Over 6: 6 delivery records, 6 legal balls
Over 7: 6 delivery records, 6 legal balls
Over 8: 6 delivery records, 6 legal balls
Over 9: 6 delivery records, 6 legal balls
Over 10: 6 delivery records, 6 legal balls
Over 11: 6 delivery records, 6 legal balls
Over 12: 6 delivery records, 6 legal balls
Over 13: 6 delivery records, 6 legal balls
Over 14: 6 delivery records, 6 legal balls
Over 15: 6 delivery records, 6 legal balls
Over 16: 6 delivery records, 6 legal balls
Over 17: 7 delivery records, 6 legal balls
Over 18: 6 delivery records, 6 legal balls
Over 19: 6 delivery records, 6 legal balls


In [37]:
# Find wickets in the first innings

wicket_count = 0

for over in first_innings["overs"]:
    for delivery in over["deliveries"]:
        if "wickets" in delivery:
            wicket_count += len(delivery["wickets"])

            print(
                "Delivery:", delivery["actual_delivery"],
                "| Batter:", delivery["batter"],
                "| Bowler:", delivery["bowler"]
            )

            for wicket in delivery["wickets"]:
                print("   Wicket:", wicket)

print("\nTotal wickets recorded:", wicket_count)

Delivery: 2.2 | Batter: Shubman Gill | Bowler: JR Hazlewood
   Wicket: {'player_out': 'Shubman Gill', 'fielders': [{'name': 'RM Patidar'}], 'kind': 'caught'}
Delivery: 3.4 | Batter: B Sai Sudharsan | Bowler: B Kumar
   Wicket: {'player_out': 'B Sai Sudharsan', 'fielders': [{'name': 'JM Sharma'}], 'kind': 'caught'}
Delivery: 7.6 | Batter: N Sindhu | Bowler: Rasikh Salam
   Wicket: {'player_out': 'N Sindhu', 'fielders': [{'name': 'D Padikkal'}], 'kind': 'caught'}
Delivery: 12.1 | Batter: JC Buttler | Bowler: KH Pandya
   Wicket: {'player_out': 'JC Buttler', 'fielders': [{'name': 'JM Sharma'}], 'kind': 'stumped'}
Delivery: 14.1 | Batter: Arshad Khan | Bowler: JR Hazlewood
   Wicket: {'player_out': 'Arshad Khan', 'fielders': [{'name': 'Rasikh Salam'}], 'kind': 'caught'}
Delivery: 16.1 | Batter: R Tewatia | Bowler: Rasikh Salam
   Wicket: {'player_out': 'R Tewatia', 'fielders': [{'name': 'RM Patidar'}], 'kind': 'caught'}
Delivery: 18.3 | Batter: JO Holder | Bowler: B Kumar
   Wicket: {'play

In [38]:
# Count dismissal types

dismissal_counts = {}

for over in first_innings["overs"]:
    for delivery in over["deliveries"]:
        for wicket in delivery.get("wickets", []):
            dismissal_type = wicket["kind"]

            dismissal_counts[dismissal_type] = (
                dismissal_counts.get(dismissal_type, 0) + 1
            )

print("Dismissal types:")

for dismissal_type, count in dismissal_counts.items():
    print(f"{dismissal_type}: {count}")

Dismissal types:
caught: 7
stumped: 1


In [39]:
# Explore the second innings

second_innings = match["innings"][1]

print("Batting team:", second_innings["team"])
print("Keys:", second_innings.keys())
print("Number of overs:", len(second_innings["overs"]))
print("Powerplays:", second_innings["powerplays"])

print("\nFirst over:")
print(second_innings["overs"][0])

Batting team: Royal Challengers Bengaluru
Keys: dict_keys(['team', 'overs', 'powerplays', 'target'])
Number of overs: 18
Powerplays: [{'from': 0.1, 'to': 5.6, 'type': 'mandatory'}]

First over:
{'over': 0, 'deliveries': [{'actual_delivery': '0.1', 'batter': 'VR Iyer', 'bowler': 'Mohammed Siraj', 'non_striker': 'V Kohli', 'replacements': {'match': [{'in': 'M Prasidh Krishna', 'out': 'R Tewatia', 'team': 'Gujarat Titans', 'reason': 'impact_player'}]}, 'runs': {'batter': 1, 'extras': 0, 'total': 1}}, {'actual_delivery': '0.2', 'batter': 'V Kohli', 'bowler': 'Mohammed Siraj', 'extras': {'legbyes': 1}, 'non_striker': 'VR Iyer', 'runs': {'batter': 0, 'extras': 1, 'total': 1}}, {'actual_delivery': '0.3', 'batter': 'VR Iyer', 'bowler': 'Mohammed Siraj', 'non_striker': 'V Kohli', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}, {'actual_delivery': '0.4', 'batter': 'VR Iyer', 'bowler': 'Mohammed Siraj', 'non_striker': 'V Kohli', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}, {'actual_delivery'

In [40]:
# Explore players listed in the match

players = match["info"]["players"]

for team, team_players in players.items():
    print(f"\n{team}")
    print("-" * len(team))
    
    for player in team_players:
        print(player)


Gujarat Titans
--------------
M Prasidh Krishna
B Sai Sudharsan
Shubman Gill
N Sindhu
JC Buttler
Washington Sundar
Arshad Khan
R Tewatia
JO Holder
Rashid Khan
K Rabada
Mohammed Siraj

Royal Challengers Bengaluru
---------------------------
JA Duffy
VR Iyer
V Kohli
D Padikkal
RM Patidar
KH Pandya
TH David
JM Sharma
R Shepherd
B Kumar
JR Hazlewood
Rasikh Salam


In [41]:
# Explore player registry

registry = match["info"]["registry"]

print("Registry keys:", registry.keys())

for key, value in registry.items():
    print(f"\n{key}:")
    print(value)

Registry keys: dict_keys(['people'])

people:
{'Arshad Khan': '12314277', 'B Kumar': '2e81a32d', 'B Sai Sudharsan': 'd5130a30', 'D Padikkal': '2c25d4f5', 'J Madanagopal': '8275b04a', 'J Srinath': 'bad31fac', 'JA Duffy': 'dadbdb68', 'JC Buttler': '99b75528', 'JM Sharma': '800d2d97', 'JO Holder': '0f721006', 'JR Hazlewood': '03806cf8', 'K Rabada': 'e62dd25d', 'KH Pandya': '5b8c830e', 'KN Ananthapadmanabhan': '3144063a', 'M Prasidh Krishna': '85e0cf10', 'Mohammed Siraj': '2f49c897', 'N Sindhu': '3cea23da', 'Nitin Menon': 'e1d41d9e', 'R Shepherd': 'c5aef772', 'R Tewatia': '39a2dfa8', 'RM Patidar': 'c740ea83', 'Rashid Khan': '5f547c8b', 'Rasikh Salam': 'b8527c3d', 'Shubman Gill': 'b4b99816', 'TH David': 'f1f99156', 'V Kohli': 'ba607b88', 'VK Sharma': 'a7a49df4', 'VR Iyer': 'a24be938', 'Washington Sundar': 'f19ccfad'}


In [42]:
# Explore wicket information

for over in match["innings"][0]["overs"]:
    for delivery in over["deliveries"]:
        
        for wicket in delivery.get("wickets", []):
            print(
                "Delivery:", delivery["actual_delivery"],
                "| Batter:", delivery["batter"],
                "| Bowler:", delivery["bowler"],
                "| Player out:", wicket["player_out"],
                "| Type:", wicket["kind"]
            )

Delivery: 2.2 | Batter: Shubman Gill | Bowler: JR Hazlewood | Player out: Shubman Gill | Type: caught
Delivery: 3.4 | Batter: B Sai Sudharsan | Bowler: B Kumar | Player out: B Sai Sudharsan | Type: caught
Delivery: 7.6 | Batter: N Sindhu | Bowler: Rasikh Salam | Player out: N Sindhu | Type: caught
Delivery: 12.1 | Batter: JC Buttler | Bowler: KH Pandya | Player out: JC Buttler | Type: stumped
Delivery: 14.1 | Batter: Arshad Khan | Bowler: JR Hazlewood | Player out: Arshad Khan | Type: caught
Delivery: 16.1 | Batter: R Tewatia | Bowler: Rasikh Salam | Player out: R Tewatia | Type: caught
Delivery: 18.3 | Batter: JO Holder | Bowler: B Kumar | Player out: JO Holder | Type: caught
Delivery: 19.2 | Batter: Rashid Khan | Bowler: Rasikh Salam | Player out: Rashid Khan | Type: caught


In [46]:
# Inspect the structure of one wicket

for over in match["innings"][0]["overs"]:
    for delivery in over["deliveries"]:
        if "wickets" in delivery:
            print(delivery["wickets"][0])
            break
    else:
        continue
    break

{'player_out': 'Shubman Gill', 'fielders': [{'name': 'RM Patidar'}], 'kind': 'caught'}


In [44]:
# Quick summary of both innings

for i, innings in enumerate(match["innings"], start=1):
    wickets = 0
    deliveries = 0

    for over in innings["overs"]:
        for delivery in over["deliveries"]:
            deliveries += 1
            wickets += len(delivery.get("wickets", []))

    print(
        f"Innings {i} | "
        f"Team: {innings['team']} | "
        f"Overs: {len(innings['overs'])} | "
        f"Delivery records: {deliveries} | "
        f"Wickets: {wickets}"
    )

Innings 1 | Team: Gujarat Titans | Overs: 20 | Delivery records: 124 | Wickets: 8
Innings 2 | Team: Royal Challengers Bengaluru | Overs: 18 | Delivery records: 109 | Wickets: 5


In [47]:
# Inspect wickets in the match

for i, innings in enumerate(match["innings"], start=1):
    print(f"\n--- Innings {i}: {innings['team']} ---")

    for over in innings["overs"]:
        for delivery in over["deliveries"]:
            wickets = delivery.get("wickets", [])

            if wickets:
                print(
                    f"Delivery: {delivery.get('actual_delivery')} | "
                    f"Batter: {delivery['batter']} | "
                    f"Bowler: {delivery['bowler']} | "
                    f"Wickets: {wickets}"
                )


--- Innings 1: Gujarat Titans ---
Delivery: 2.2 | Batter: Shubman Gill | Bowler: JR Hazlewood | Wickets: [{'player_out': 'Shubman Gill', 'fielders': [{'name': 'RM Patidar'}], 'kind': 'caught'}]
Delivery: 3.4 | Batter: B Sai Sudharsan | Bowler: B Kumar | Wickets: [{'player_out': 'B Sai Sudharsan', 'fielders': [{'name': 'JM Sharma'}], 'kind': 'caught'}]
Delivery: 7.6 | Batter: N Sindhu | Bowler: Rasikh Salam | Wickets: [{'player_out': 'N Sindhu', 'fielders': [{'name': 'D Padikkal'}], 'kind': 'caught'}]
Delivery: 12.1 | Batter: JC Buttler | Bowler: KH Pandya | Wickets: [{'player_out': 'JC Buttler', 'fielders': [{'name': 'JM Sharma'}], 'kind': 'stumped'}]
Delivery: 14.1 | Batter: Arshad Khan | Bowler: JR Hazlewood | Wickets: [{'player_out': 'Arshad Khan', 'fielders': [{'name': 'Rasikh Salam'}], 'kind': 'caught'}]
Delivery: 16.1 | Batter: R Tewatia | Bowler: Rasikh Salam | Wickets: [{'player_out': 'R Tewatia', 'fielders': [{'name': 'RM Patidar'}], 'kind': 'caught'}]
Delivery: 18.3 | Batter

In [48]:
# Count dismissal types in the match

dismissal_counts = {}

for innings in match["innings"]:
    for over in innings["overs"]:
        for delivery in over["deliveries"]:
            for wicket in delivery.get("wickets", []):
                kind = wicket["kind"]
                dismissal_counts[kind] = dismissal_counts.get(kind, 0) + 1

print("Dismissal types:")
for kind, count in dismissal_counts.items():
    print(f"{kind}: {count}")

Dismissal types:
caught: 11
stumped: 1
lbw: 1


In [49]:
# Check whether any delivery contains multiple wickets

multiple_wicket_deliveries = []

for i, innings in enumerate(match["innings"], start=1):
    for over in innings["overs"]:
        for delivery in over["deliveries"]:
            wickets = delivery.get("wickets", [])

            if len(wickets) > 1:
                multiple_wicket_deliveries.append(
                    {
                        "innings": i,
                        "delivery": delivery["actual_delivery"],
                        "wickets": wickets
                    }
                )

print("Deliveries with multiple wickets:", len(multiple_wicket_deliveries))

for item in multiple_wicket_deliveries:
    print(item)

Deliveries with multiple wickets: 0


## Data Model Design

### Match Table

**Grain:** One row per match.

The Match Table stores match-level information such as date, teams, venue, toss, result, and player of the match.

### Delivery Table

**Grain:** One row per delivery event.

The Delivery Table stores ball-by-ball information such as innings, batting team, bowler, batter, runs, extras, and wickets.

### Relationship

One match can contain many delivery records.

Therefore:

**Match Table (1) → Delivery Table (many)**

### Important Data Concepts

A delivery record is not always the same as a legal ball because wides and no-balls can create additional delivery records.

Wicket information is nested inside a delivery and must be flattened into analysis-friendly columns.

In [52]:
# Extract match-level information

info = match["info"]

match_id = "1535465"

date = info["dates"][0]
season = info["season"]
competition = info["event"]["name"]
stage = info["event"].get("stage")

venue = info["venue"]
city = info.get("city")

team_1 = info["teams"][0]
team_2 = info["teams"][1]

toss_winner = info["toss"]["winner"]
toss_decision = info["toss"]["decision"]

winner = info["outcome"]["winner"]

player_of_match = ", ".join(info["player_of_match"])

print("Match information extracted successfully.")

Match information extracted successfully.


In [53]:
# Extract winning margin

win_by = info["outcome"]["by"]

if "runs" in win_by:
    win_type = "runs"
    win_margin = win_by["runs"]

elif "wickets" in win_by:
    win_type = "wickets"
    win_margin = win_by["wickets"]

else:
    win_type = None
    win_margin = None

print("Win type:", win_type)
print("Win margin:", win_margin)

Win type: wickets
Win margin: 5


In [54]:
import pandas as pd

match_df = pd.DataFrame([{
    "match_id": match_id,
    "date": date,
    "season": season,
    "competition": competition,
    "stage": stage,
    "venue": venue,
    "city": city,
    "team_1": team_1,
    "team_2": team_2,
    "toss_winner": toss_winner,
    "toss_decision": toss_decision,
    "winner": winner,
    "win_type": win_type,
    "win_margin": win_margin,
    "player_of_match": player_of_match
}])

match_df

,match_id,date,season,competition,stage,venue,city,team_1,team_2,toss_winner,toss_decision,winner,win_type,win_margin,player_of_match
0,1535465,2026-05-31,2026,Indian Premier League,Final,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,Gujarat Titans,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Royal Challengers Bengaluru,wickets,5,V Kohli


In [55]:
# Basic validation of Match Table

print("Rows:", len(match_df))
print("Columns:", len(match_df.columns))
print("\nColumn names:")
print(match_df.columns.tolist())

print("\nData types:")
print(match_df.dtypes)

print("\nMatch Table:")
print(match_df.to_string(index=False))

Rows: 1
Columns: 15

Column names:
['match_id', 'date', 'season', 'competition', 'stage', 'venue', 'city', 'team_1', 'team_2', 'toss_winner', 'toss_decision', 'winner', 'win_type', 'win_margin', 'player_of_match']

Data types:
match_id             str
date                 str
season               str
competition          str
stage                str
venue                str
city                 str
team_1               str
team_2               str
toss_winner          str
toss_decision        str
winner               str
win_type             str
win_margin         int64
player_of_match      str
dtype: object

Match Table:
match_id       date season           competition stage                            venue      city         team_1                      team_2                 toss_winner toss_decision                      winner win_type  win_margin player_of_match
 1535465 2026-05-31   2026 Indian Premier League Final Narendra Modi Stadium, Ahmedabad Ahmedabad Gujarat Titans Royal Cha

In [56]:
# Extract delivery-level records

delivery_records = []

for innings_number, innings in enumerate(match["innings"], start=1):

    batting_team = innings["team"]

    bowling_team = next(
        team for team in info["teams"]
        if team != batting_team
    )

    for over in innings["overs"]:

        over_number = over["over"]

        for delivery in over["deliveries"]:

            delivery_records.append({
                "match_id": match_id,
                "innings": innings_number,
                "batting_team": batting_team,
                "bowling_team": bowling_team,
                "over": over_number,
                "actual_delivery": delivery["actual_delivery"],
                "batter": delivery["batter"],
                "non_striker": delivery["non_striker"],
                "bowler": delivery["bowler"],
                "runs_batter": delivery["runs"]["batter"],
                "runs_extras": delivery["runs"]["extras"],
                "runs_total": delivery["runs"]["total"]
            })

print("Delivery records extracted:", len(delivery_records))

Delivery records extracted: 233


In [58]:
# Add useful delivery flags

for row in delivery_records:

    # Extras
    # We inspect the original delivery again using its position later.
    row["is_wide"] = False
    row["is_no_ball"] = False
    row["is_wicket"] = False
    # Create initial Delivery Table

delivery_df = pd.DataFrame(delivery_records)

delivery_df.head(10)

,match_id,innings,batting_team,bowling_team,over,actual_delivery,batter,non_striker,bowler,runs_batter,runs_extras,runs_total,is_wide,is_no_ball,is_wicket
0,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.1,B Sai Sudharsan,Shubman Gill,JA Duffy,0,0,0,False,False,False
1,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.2,B Sai Sudharsan,Shubman Gill,JA Duffy,0,0,0,False,False,False
2,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.3,B Sai Sudharsan,Shubman Gill,JA Duffy,4,0,4,False,False,False
3,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.4,B Sai Sudharsan,Shubman Gill,JA Duffy,4,0,4,False,False,False
4,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.5,B Sai Sudharsan,Shubman Gill,JA Duffy,0,1,1,False,False,False
5,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.5,B Sai Sudharsan,Shubman Gill,JA Duffy,0,1,1,False,False,False
6,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.5,B Sai Sudharsan,Shubman Gill,JA Duffy,1,0,1,False,False,False
7,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.6,Shubman Gill,B Sai Sudharsan,JA Duffy,0,1,1,False,False,False
8,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.6,Shubman Gill,B Sai Sudharsan,JA Duffy,1,0,1,False,False,False
9,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,1,1.1,Shubman Gill,B Sai Sudharsan,B Kumar,0,0,0,False,False,False


In [59]:
# Validate Delivery Table

print("Rows:", len(delivery_df))
print("Columns:", len(delivery_df.columns))

print("\nColumns:")
print(delivery_df.columns.tolist())

print("\nData types:")
print(delivery_df.dtypes)

print("\nFirst 5 rows:")
print(delivery_df.head().to_string(index=False))

Rows: 233
Columns: 15

Columns:
['match_id', 'innings', 'batting_team', 'bowling_team', 'over', 'actual_delivery', 'batter', 'non_striker', 'bowler', 'runs_batter', 'runs_extras', 'runs_total', 'is_wide', 'is_no_ball', 'is_wicket']

Data types:
match_id             str
innings            int64
batting_team         str
bowling_team         str
over               int64
actual_delivery      str
batter               str
non_striker          str
bowler               str
runs_batter        int64
runs_extras        int64
runs_total         int64
is_wide             bool
is_no_ball          bool
is_wicket           bool
dtype: object

First 5 rows:
match_id  innings   batting_team                bowling_team  over actual_delivery          batter  non_striker   bowler  runs_batter  runs_extras  runs_total  is_wide  is_no_ball  is_wicket
 1535465        1 Gujarat Titans Royal Challengers Bengaluru     0             0.1 B Sai Sudharsan Shubman Gill JA Duffy            0            0           0  

In [60]:
# Rebuild Delivery Table with extras and wicket information

delivery_records = []

for innings_number, innings in enumerate(match["innings"], start=1):

    batting_team = innings["team"]

    bowling_team = next(
        team for team in info["teams"]
        if team != batting_team
    )

    for over in innings["overs"]:

        for delivery in over["deliveries"]:

            extras = delivery.get("extras", {})
            wickets = delivery.get("wickets", [])

            is_wide = "wides" in extras
            is_no_ball = "noballs" in extras
            is_wicket = len(wickets) > 0

            delivery_records.append({
                "match_id": match_id,
                "innings": innings_number,
                "batting_team": batting_team,
                "bowling_team": bowling_team,
                "over": over["over"],
                "actual_delivery": delivery["actual_delivery"],
                "batter": delivery["batter"],
                "non_striker": delivery["non_striker"],
                "bowler": delivery["bowler"],
                "runs_batter": delivery["runs"]["batter"],
                "runs_extras": delivery["runs"]["extras"],
                "runs_total": delivery["runs"]["total"],
                "is_wide": is_wide,
                "is_no_ball": is_no_ball,
                "is_wicket": is_wicket
            })

delivery_df = pd.DataFrame(delivery_records)

print("Delivery records:", len(delivery_df))

Delivery records: 233


In [61]:
# Validate extras and wicket flags

print("Wide deliveries:", delivery_df["is_wide"].sum())
print("No-ball deliveries:", delivery_df["is_no_ball"].sum())
print("Wicket deliveries:", delivery_df["is_wicket"].sum())

print("\nExtra deliveries:")
print(
    delivery_df[
        (delivery_df["is_wide"]) |
        (delivery_df["is_no_ball"])
    ][
        ["innings", "actual_delivery", "batter",
         "bowler", "runs_extras", "is_wide", "is_no_ball"]
    ].to_string(index=False)
)

Wide deliveries: 5
No-ball deliveries: 0
Wicket deliveries: 13

Extra deliveries:
 innings actual_delivery            batter         bowler  runs_extras  is_wide  is_no_ball
       1             0.5   B Sai Sudharsan       JA Duffy            1     True       False
       1             0.5   B Sai Sudharsan       JA Duffy            1     True       False
       1             0.6      Shubman Gill       JA Duffy            1     True       False
       1            17.5 Washington Sundar   JR Hazlewood            1     True       False
       2             6.6        RM Patidar Mohammed Siraj            1     True       False


In [62]:
# A wide or no-ball does not count as a legal delivery

delivery_df["is_legal_delivery"] = (
    ~delivery_df["is_wide"] &
    ~delivery_df["is_no_ball"]
)

print("Total delivery events:", len(delivery_df))
print("Legal deliveries:", delivery_df["is_legal_delivery"].sum())
print("Non-legal deliveries:", (~delivery_df["is_legal_delivery"]).sum())

Total delivery events: 233
Legal deliveries: 228
Non-legal deliveries: 5


In [63]:
# Inspect delivery events around extras

delivery_df[
    [
        "innings",
        "over",
        "actual_delivery",
        "batter",
        "bowler",
        "runs_batter",
        "runs_extras",
        "runs_total",
        "is_wide",
        "is_no_ball",
        "is_legal_delivery",
        "is_wicket"
    ]
].head(20)

,innings,over,actual_delivery,batter,bowler,runs_batter,runs_extras,runs_total,is_wide,is_no_ball,is_legal_delivery,is_wicket
0,1,0,0.1,B Sai Sudharsan,JA Duffy,0,0,0,False,False,True,False
1,1,0,0.2,B Sai Sudharsan,JA Duffy,0,0,0,False,False,True,False
2,1,0,0.3,B Sai Sudharsan,JA Duffy,4,0,4,False,False,True,False
3,1,0,0.4,B Sai Sudharsan,JA Duffy,4,0,4,False,False,True,False
4,1,0,0.5,B Sai Sudharsan,JA Duffy,0,1,1,True,False,False,False
5,1,0,0.5,B Sai Sudharsan,JA Duffy,0,1,1,True,False,False,False
6,1,0,0.5,B Sai Sudharsan,JA Duffy,1,0,1,False,False,True,False
7,1,0,0.6,Shubman Gill,JA Duffy,0,1,1,True,False,False,False
8,1,0,0.6,Shubman Gill,JA Duffy,1,0,1,False,False,True,False
9,1,1,1.1,Shubman Gill,B Kumar,0,0,0,False,False,True,False


In [64]:
# Inspect delivery events around extras

delivery_df[
    [
        "innings",
        "over",
        "actual_delivery",
        "batter",
        "bowler",
        "runs_batter",
        "runs_extras",
        "runs_total",
        "is_wide",
        "is_no_ball",
        "is_legal_delivery",
        "is_wicket"
    ]
].head(20)

,innings,over,actual_delivery,batter,bowler,runs_batter,runs_extras,runs_total,is_wide,is_no_ball,is_legal_delivery,is_wicket
0,1,0,0.1,B Sai Sudharsan,JA Duffy,0,0,0,False,False,True,False
1,1,0,0.2,B Sai Sudharsan,JA Duffy,0,0,0,False,False,True,False
2,1,0,0.3,B Sai Sudharsan,JA Duffy,4,0,4,False,False,True,False
3,1,0,0.4,B Sai Sudharsan,JA Duffy,4,0,4,False,False,True,False
4,1,0,0.5,B Sai Sudharsan,JA Duffy,0,1,1,True,False,False,False
5,1,0,0.5,B Sai Sudharsan,JA Duffy,0,1,1,True,False,False,False
6,1,0,0.5,B Sai Sudharsan,JA Duffy,1,0,1,False,False,True,False
7,1,0,0.6,Shubman Gill,JA Duffy,0,1,1,True,False,False,False
8,1,0,0.6,Shubman Gill,JA Duffy,1,0,1,False,False,True,False
9,1,1,1.1,Shubman Gill,B Kumar,0,0,0,False,False,True,False


In [66]:
# Add wicket details safely

delivery_df["player_out"] = None
delivery_df["dismissal_type"] = None

row_index = 0

for innings in match["innings"]:

    for over in innings["overs"]:

        for delivery in over["deliveries"]:

            wickets = delivery.get("wickets", [])

            if wickets:
                wicket = wickets[0]

                delivery_df.loc[row_index, "player_out"] = wicket["player_out"]
                delivery_df.loc[row_index, "dismissal_type"] = wicket["kind"]

            row_index += 1

print("Wicket details added.")

Wicket details added.


In [67]:
# Inspect all wicket records

wicket_df = delivery_df[delivery_df["is_wicket"]].copy()

print("Wicket rows:", len(wicket_df))

print(
    wicket_df[
        [
            "innings",
            "actual_delivery",
            "batter",
            "bowler",
            "player_out",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

Wicket rows: 13
 innings actual_delivery          batter         bowler      player_out dismissal_type
       1             2.2    Shubman Gill   JR Hazlewood    Shubman Gill         caught
       1             3.4 B Sai Sudharsan        B Kumar B Sai Sudharsan         caught
       1             7.6        N Sindhu   Rasikh Salam        N Sindhu         caught
       1            12.1      JC Buttler      KH Pandya      JC Buttler        stumped
       1            14.1     Arshad Khan   JR Hazlewood     Arshad Khan         caught
       1            16.1       R Tewatia   Rasikh Salam       R Tewatia         caught
       1            18.3       JO Holder        B Kumar       JO Holder         caught
       1            19.2     Rashid Khan   Rasikh Salam     Rashid Khan         caught
       2             4.3         VR Iyer Mohammed Siraj         VR Iyer         caught
       2             5.1      D Padikkal       K Rabada      D Padikkal         caught
       2             8.2   

In [68]:
# Add useful performance flags

delivery_df["is_dot_ball"] = (
    (delivery_df["runs_total"] == 0) &
    delivery_df["is_legal_delivery"]
)

delivery_df["is_boundary"] = (
    delivery_df["runs_batter"].isin([4, 6])
)

delivery_df["is_four"] = (
    delivery_df["runs_batter"] == 4
)

delivery_df["is_six"] = (
    delivery_df["runs_batter"] == 6
)

print("Dot balls:", delivery_df["is_dot_ball"].sum())
print("Boundaries:", delivery_df["is_boundary"].sum())
print("Fours:", delivery_df["is_four"].sum())
print("Sixes:", delivery_df["is_six"].sum())

Dot balls: 80
Boundaries: 43
Fours: 33
Sixes: 10


In [69]:
# Add T20 match phase

def get_phase(over):
    if over < 6:
        return "Powerplay"
    elif over < 15:
        return "Middle"
    else:
        return "Death"


delivery_df["phase"] = delivery_df["over"].apply(get_phase)

print(
    delivery_df["phase"]
    .value_counts()
    .sort_index()
)

phase
Death         49
Middle       109
Powerplay     75
Name: count, dtype: int64


In [71]:
# Calculate innings totals from Delivery Table

innings_summary = (
    delivery_df
    .groupby(["innings", "batting_team"])
    .agg(
        total_runs=("runs_total", "sum"),
        wickets=("is_wicket", "sum"),
        delivery_events=("match_id", "size"),
        legal_deliveries=("is_legal_delivery", "sum")
    )
    .reset_index()
)

innings_summary

,innings,batting_team,total_runs,wickets,delivery_events,legal_deliveries
0,1,Gujarat Titans,155,8,124,120
1,2,Royal Challengers Bengaluru,161,5,109,108


In [1]:
# Convert legal deliveries into completed overs + remaining balls

for _, row in innings_summary.iterrows():

    legal_balls = row["legal_deliveries"]

    completed_overs = legal_balls // 6
    remaining_balls = legal_balls % 6

    print(
        f"Innings {row['innings']} | "
        f"{row['batting_team']} | "
        f"Legal balls: {legal_balls} | "
        f"Overs: {completed_overs}.{remaining_balls}"
    )

NameError: name 'innings_summary' is not defined

In [73]:
# Validate wicket count

print("Wickets by innings:")

print(
    delivery_df
    .groupby(["innings", "batting_team"])["is_wicket"]
    .sum()
)

print("\nTotal wickets:", delivery_df["is_wicket"].sum())

Wickets by innings:
innings  batting_team               
1        Gujarat Titans                 8
2        Royal Challengers Bengaluru    5
Name: is_wicket, dtype: int64

Total wickets: 13


In [74]:
# Final Delivery Table validation

print("=== DELIVERY TABLE VALIDATION ===")

print("Rows:", len(delivery_df))
print("Delivery events:", len(delivery_df))
print("Legal deliveries:", delivery_df["is_legal_delivery"].sum())
print("Wickets:", delivery_df["is_wicket"].sum())
print("Runs:", delivery_df["runs_total"].sum())

print("\nMissing values:")
print(delivery_df.isna().sum())

=== DELIVERY TABLE VALIDATION ===
Rows: 233
Delivery events: 233
Legal deliveries: 228
Wickets: 13
Runs: 316

Missing values:
match_id               0
innings                0
batting_team           0
bowling_team           0
over                   0
actual_delivery        0
batter                 0
non_striker            0
bowler                 0
runs_batter            0
runs_extras            0
runs_total             0
is_wide                0
is_no_ball             0
is_wicket              0
is_legal_delivery      0
player_out           220
dismissal_type       220
is_dot_ball            0
is_boundary            0
is_four                0
is_six                 0
phase                  0
dtype: int64


In [76]:
# Check for duplicate delivery records

duplicate_columns = [
    "match_id",
    "innings",
    "over",
    "actual_delivery",
    "batter",
    "bowler"
]

duplicates = delivery_df[
    delivery_df.duplicated(
        subset=duplicate_columns,
        keep=False
    )
]

print("Duplicate delivery rows:", len(duplicates))

if len(duplicates) > 0:
    print("\nPotential duplicates:")
    print(duplicates[duplicate_columns].to_string(index=False))
else:
    print("PASS: No duplicate delivery records found.")

Duplicate delivery rows: 9

Potential duplicates:
match_id  innings  over actual_delivery            batter         bowler
 1535465        1     0             0.5   B Sai Sudharsan       JA Duffy
 1535465        1     0             0.5   B Sai Sudharsan       JA Duffy
 1535465        1     0             0.5   B Sai Sudharsan       JA Duffy
 1535465        1     0             0.6      Shubman Gill       JA Duffy
 1535465        1     0             0.6      Shubman Gill       JA Duffy
 1535465        1    17            17.5 Washington Sundar   JR Hazlewood
 1535465        1    17            17.5 Washington Sundar   JR Hazlewood
 1535465        2     6             6.6        RM Patidar Mohammed Siraj
 1535465        2     6             6.6        RM Patidar Mohammed Siraj


In [77]:
# Check critical columns for missing values

required_columns = [
    "match_id",
    "innings",
    "batting_team",
    "bowling_team",
    "over",
    "actual_delivery",
    "batter",
    "non_striker",
    "bowler",
    "runs_batter",
    "runs_extras",
    "runs_total",
    "is_legal_delivery"
]

missing_check = delivery_df[required_columns].isna().sum()

print("Missing values in critical columns:\n")
print(missing_check)

if missing_check.sum() == 0:
    print("\nPASS: No missing values in critical columns.")
else:
    print("\nCHECK REQUIRED: Missing values found.")

Missing values in critical columns:

match_id             0
innings              0
batting_team         0
bowling_team         0
over                 0
actual_delivery      0
batter               0
non_striker          0
bowler               0
runs_batter          0
runs_extras          0
runs_total           0
is_legal_delivery    0
dtype: int64

PASS: No missing values in critical columns.


In [78]:
# Check innings structure

print("=== INNINGS STRUCTURE ===")

for innings_number, group in delivery_df.groupby("innings"):

    print(f"\nInnings {innings_number}")
    print("Batting team:", group["batting_team"].iloc[0])
    print("Bowling team:", group["bowling_team"].iloc[0])
    print("Overs present:", group["over"].min(), "to", group["over"].max())
    print("Delivery events:", len(group))
    print("Legal deliveries:", group["is_legal_delivery"].sum())
    print("Runs:", group["runs_total"].sum())
    print("Wickets:", group["is_wicket"].sum())

=== INNINGS STRUCTURE ===

Innings 1
Batting team: Gujarat Titans
Bowling team: Royal Challengers Bengaluru
Overs present: 0 to 19
Delivery events: 124
Legal deliveries: 120
Runs: 155
Wickets: 8

Innings 2
Batting team: Royal Challengers Bengaluru
Bowling team: Gujarat Titans
Overs present: 0 to 17
Delivery events: 109
Legal deliveries: 108
Runs: 161
Wickets: 5


In [79]:
# Create a reusable validation report

validation_report = {
    "delivery_rows": len(delivery_df),
    "duplicate_rows": len(duplicates),
    "missing_critical_values": int(missing_check.sum()),
    "total_runs": int(delivery_df["runs_total"].sum()),
    "total_wickets": int(delivery_df["is_wicket"].sum()),
    "legal_deliveries": int(delivery_df["is_legal_delivery"].sum()),
    "wide_deliveries": int(delivery_df["is_wide"].sum()),
    "no_ball_deliveries": int(delivery_df["is_no_ball"].sum())
}

validation_report

{'delivery_rows': 233,
 'duplicate_rows': 9,
 'missing_critical_values': 0,
 'total_runs': 316,
 'total_wickets': 13,
 'legal_deliveries': 228,
 'wide_deliveries': 5,
 'no_ball_deliveries': 0}

In [80]:
from pathlib import Path

raw_data_path = Path("data/raw")

json_files = list(raw_data_path.glob("*.json"))

print("JSON files found:", len(json_files))

for file in json_files[:10]:
    print(file)

JSON files found: 1
data\raw\1535465.json


In [81]:
import json

def load_match_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)


# Test the function using our current match
test_match = load_match_json(json_files[0])

print("Loaded:", json_files[0].name)
print("Top-level keys:", list(test_match.keys()))

Loaded: 1535465.json
Top-level keys: ['meta', 'info', 'innings']


In [82]:
all_matches = []

for file_path in json_files:

    match_data = load_match_json(file_path)

    all_matches.append({
        "file_name": file_path.name,
        "match_data": match_data
    })

print("Matches loaded:", len(all_matches))

print("\nFirst match:")
print(all_matches[0]["file_name"])

Matches loaded: 1

First match:
1535465.json


In [83]:
for item in all_matches:

    match_data = item["match_data"]
    info = match_data["info"]

    print(
        f"{item['file_name']} | "
        f"{info['dates'][0]} | "
        f"{info['teams'][0]} vs {info['teams'][1]}"
    )

1535465.json | 2026-05-31 | Gujarat Titans vs Royal Challengers Bengaluru


In [84]:
# Function to extract one Match Table row

def parse_match(match_data, match_id):
    info = match_data["info"]

    outcome = info.get("outcome", {})
    winner = outcome.get("winner")
    win_by = outcome.get("by", {})

    if "runs" in win_by:
        win_type = "runs"
        win_margin = win_by["runs"]
    elif "wickets" in win_by:
        win_type = "wickets"
        win_margin = win_by["wickets"]
    else:
        win_type = None
        win_margin = None

    return {
        "match_id": match_id,
        "date": info["dates"][0],
        "season": info["season"],
        "competition": info.get("event", {}).get("name"),
        "stage": info.get("event", {}).get("stage"),
        "venue": info.get("venue"),
        "city": info.get("city"),
        "team_1": info["teams"][0],
        "team_2": info["teams"][1],
        "toss_winner": info["toss"]["winner"],
        "toss_decision": info["toss"]["decision"],
        "winner": winner,
        "win_type": win_type,
        "win_margin": win_margin,
        "player_of_match": ", ".join(info.get("player_of_match", []))
    }

In [85]:
# Test the reusable Match parser

test_match_row = parse_match(
    test_match,
    json_files[0].stem
)

test_match_row


{'match_id': '1535465',
 'date': '2026-05-31',
 'season': '2026',
 'competition': 'Indian Premier League',
 'stage': 'Final',
 'venue': 'Narendra Modi Stadium, Ahmedabad',
 'city': 'Ahmedabad',
 'team_1': 'Gujarat Titans',
 'team_2': 'Royal Challengers Bengaluru',
 'toss_winner': 'Royal Challengers Bengaluru',
 'toss_decision': 'field',
 'winner': 'Royal Challengers Bengaluru',
 'win_type': 'wickets',
 'win_margin': 5,
 'player_of_match': 'V Kohli'}

In [86]:
# Function to extract delivery records from one match

def parse_deliveries(match_data, match_id):

    info = match_data["info"]
    delivery_records = []

    for innings_number, innings in enumerate(match_data["innings"], start=1):

        batting_team = innings["team"]

        bowling_team = next(
            team for team in info["teams"]
            if team != batting_team
        )

        for over in innings["overs"]:

            for delivery in over["deliveries"]:

                extras = delivery.get("extras", {})
                wickets = delivery.get("wickets", [])

                is_wide = "wides" in extras
                is_no_ball = "noballs" in extras
                is_wicket = len(wickets) > 0

                player_out = None
                dismissal_type = None

                if wickets:
                    player_out = wickets[0]["player_out"]
                    dismissal_type = wickets[0]["kind"]

                is_legal_delivery = (
                    not is_wide and
                    not is_no_ball
                )

                runs_batter = delivery["runs"]["batter"]
                runs_extras = delivery["runs"]["extras"]
                runs_total = delivery["runs"]["total"]

                delivery_records.append({
                    "match_id": match_id,
                    "innings": innings_number,
                    "batting_team": batting_team,
                    "bowling_team": bowling_team,
                    "over": over["over"],
                    "actual_delivery": delivery["actual_delivery"],
                    "batter": delivery["batter"],
                    "non_striker": delivery["non_striker"],
                    "bowler": delivery["bowler"],
                    "runs_batter": runs_batter,
                    "runs_extras": runs_extras,
                    "runs_total": runs_total,
                    "is_wide": is_wide,
                    "is_no_ball": is_no_ball,
                    "is_legal_delivery": is_legal_delivery,
                    "is_dot_ball": (
                        runs_total == 0 and is_legal_delivery
                    ),
                    "is_boundary": runs_batter in [4, 6],
                    "is_four": runs_batter == 4,
                    "is_six": runs_batter == 6,
                    "is_wicket": is_wicket,
                    "player_out": player_out,
                    "dismissal_type": dismissal_type,
                    "phase": get_phase(over["over"])
                })

    return delivery_records

In [87]:
# Test the reusable Delivery parser

test_delivery_records = parse_deliveries(
    test_match,
    json_files[0].stem
)

test_delivery_df = pd.DataFrame(test_delivery_records)

print("Rows:", len(test_delivery_df))
print("Columns:", len(test_delivery_df.columns))

test_delivery_df.head()

Rows: 233
Columns: 23


,match_id,innings,batting_team,bowling_team,over,actual_delivery,batter,non_striker,bowler,runs_batter,...,is_no_ball,is_legal_delivery,is_dot_ball,is_boundary,is_four,is_six,is_wicket,player_out,dismissal_type,phase
0,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.1,B Sai Sudharsan,Shubman Gill,JA Duffy,0,...,False,True,True,False,False,False,False,NaN,NaN,Powerplay
1,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.2,B Sai Sudharsan,Shubman Gill,JA Duffy,0,...,False,True,True,False,False,False,False,NaN,NaN,Powerplay
2,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.3,B Sai Sudharsan,Shubman Gill,JA Duffy,4,...,False,True,False,True,True,False,False,NaN,NaN,Powerplay
3,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.4,B Sai Sudharsan,Shubman Gill,JA Duffy,4,...,False,True,False,True,True,False,False,NaN,NaN,Powerplay
4,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.5,B Sai Sudharsan,Shubman Gill,JA Duffy,0,...,False,False,False,False,False,False,False,NaN,NaN,Powerplay


In [88]:
# Parse all JSON files into Match records

all_match_records = []

for file_path in json_files:

    match_data = load_match_json(file_path)

    match_id = file_path.stem

    match_row = parse_match(
        match_data,
        match_id
    )

    all_match_records.append(match_row)

matches_df = pd.DataFrame(all_match_records)

print("Matches processed:", len(matches_df))
print("Match table rows:", len(matches_df))

matches_df.head()

Matches processed: 1
Match table rows: 1


,match_id,date,season,competition,stage,venue,city,team_1,team_2,toss_winner,toss_decision,winner,win_type,win_margin,player_of_match
0,1535465,2026-05-31,2026,Indian Premier League,Final,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,Gujarat Titans,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Royal Challengers Bengaluru,wickets,5,V Kohli


In [89]:
# Parse all JSON files into Delivery records

all_delivery_records = []

for file_path in json_files:

    match_data = load_match_json(file_path)

    match_id = file_path.stem

    match_delivery_records = parse_deliveries(
        match_data,
        match_id
    )

    all_delivery_records.extend(
        match_delivery_records
    )

deliveries_df = pd.DataFrame(all_delivery_records)

print("Delivery records processed:", len(deliveries_df))
print("Delivery table rows:", len(deliveries_df))

deliveries_df.head()

Delivery records processed: 233
Delivery table rows: 233


,match_id,innings,batting_team,bowling_team,over,actual_delivery,batter,non_striker,bowler,runs_batter,...,is_no_ball,is_legal_delivery,is_dot_ball,is_boundary,is_four,is_six,is_wicket,player_out,dismissal_type,phase
0,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.1,B Sai Sudharsan,Shubman Gill,JA Duffy,0,...,False,True,True,False,False,False,False,NaN,NaN,Powerplay
1,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.2,B Sai Sudharsan,Shubman Gill,JA Duffy,0,...,False,True,True,False,False,False,False,NaN,NaN,Powerplay
2,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.3,B Sai Sudharsan,Shubman Gill,JA Duffy,4,...,False,True,False,True,True,False,False,NaN,NaN,Powerplay
3,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.4,B Sai Sudharsan,Shubman Gill,JA Duffy,4,...,False,True,False,True,True,False,False,NaN,NaN,Powerplay
4,1535465,1,Gujarat Titans,Royal Challengers Bengaluru,0,0.5,B Sai Sudharsan,Shubman Gill,JA Duffy,0,...,False,False,False,False,False,False,False,NaN,NaN,Powerplay


In [90]:
# Compare reusable pipeline output with our earlier tables

print("=== MATCH TABLE ===")
print("Old rows:", len(match_df))
print("New rows:", len(matches_df))

print("\n=== DELIVERY TABLE ===")
print("Old rows:", len(delivery_df))
print("New rows:", len(deliveries_df))

print("\nDelivery row difference:",
      len(deliveries_df) - len(delivery_df))

=== MATCH TABLE ===
Old rows: 1
New rows: 1

=== DELIVERY TABLE ===
Old rows: 233
New rows: 233

Delivery row difference: 0


In [91]:
# Basic integrity check of the new master tables

print("=== MASTER TABLE CHECK ===")

print("Matches:", len(matches_df))
print("Delivery records:", len(deliveries_df))

print("\nUnique match IDs in Match Table:",
      matches_df["match_id"].nunique())

print("Unique match IDs in Delivery Table:",
      deliveries_df["match_id"].nunique())

print("\nDelivery records by match:")
print(
    deliveries_df
    .groupby("match_id")
    .size()
)

=== MASTER TABLE CHECK ===
Matches: 1
Delivery records: 233

Unique match IDs in Match Table: 1
Unique match IDs in Delivery Table: 1

Delivery records by match:
match_id
1535465    233
dtype: int64


In [92]:
# Safely process one Cricsheet JSON file

def process_match_file(file_path):

    try:
        match_data = load_match_json(file_path)
        match_id = file_path.stem

        match_row = parse_match(
            match_data,
            match_id
        )

        delivery_records = parse_deliveries(
            match_data,
            match_id
        )

        return {
            "success": True,
            "file_name": file_path.name,
            "match_row": match_row,
            "delivery_records": delivery_records,
            "error": None
        }

    except Exception as e:

        return {
            "success": False,
            "file_name": file_path.name,
            "match_row": None,
            "delivery_records": [],
            "error": str(e)
        }

In [93]:
# Process every JSON file safely

processed_matches = []
failed_matches = []

for file_path in json_files:

    result = process_match_file(file_path)

    if result["success"]:
        processed_matches.append(result)
    else:
        failed_matches.append(result)

print("Successful files:", len(processed_matches))
print("Failed files:", len(failed_matches))

Successful files: 1
Failed files: 0


In [94]:
# Inspect any files that failed

if failed_matches:

    for result in failed_matches:
        print(
            f"FAILED: {result['file_name']} | "
            f"Error: {result['error']}"
        )

else:

    print("PASS: No files failed during ingestion.")

PASS: No files failed during ingestion.


In [95]:
# Build master tables from successfully processed matches

matches_df = pd.DataFrame(
    [
        result["match_row"]
        for result in processed_matches
    ]
)

all_delivery_records = []

for result in processed_matches:
    all_delivery_records.extend(
        result["delivery_records"]
    )

deliveries_df = pd.DataFrame(
    all_delivery_records
)

print("=== INGESTION RESULT ===")
print("Matches:", len(matches_df))
print("Delivery records:", len(deliveries_df))

=== INGESTION RESULT ===
Matches: 1
Delivery records: 233


In [96]:
# Check current raw data before adding the full dataset

from pathlib import Path

raw_data_path = Path("data/raw")

json_files = list(raw_data_path.glob("*.json"))

print("Current JSON match files:", len(json_files))
print("\nFiles:")

for file_path in json_files[:10]:
    print(file_path.name)

Current JSON match files: 1

Files:
1535465.json


In [97]:
# Create directories for processed data

processed_data_path = Path("data/processed")
analytics_data_path = Path("data/analytics")

processed_data_path.mkdir(parents=True, exist_ok=True)
analytics_data_path.mkdir(parents=True, exist_ok=True)

print("Processed data directory:", processed_data_path)
print("Analytics data directory:", analytics_data_path)

Processed data directory: data\processed
Analytics data directory: data\analytics


In [98]:
# Define the initial project dataset scope

DATA_SOURCE = "Cricsheet"
COMPETITION = "Indian Premier League"

print("Data source:", DATA_SOURCE)
print("Competition:", COMPETITION)
print("Initial scope: IPL")

Data source: Cricsheet
Competition: Indian Premier League
Initial scope: IPL


In [99]:
# Confirm the ingestion pipeline is ready for scaling

print("=== CURRENT PIPELINE STATUS ===")

print("JSON files available:", len(json_files))
print("Matches processed:", len(matches_df))
print("Delivery records:", len(deliveries_df))

print("\nPipeline components ready:")
print("✓ JSON loader")
print("✓ Match parser")
print("✓ Delivery parser")
print("✓ Safe file processing")
print("✓ Match Table")
print("✓ Delivery Table")

=== CURRENT PIPELINE STATUS ===
JSON files available: 1
Matches processed: 1
Delivery records: 233

Pipeline components ready:
✓ JSON loader
✓ Match parser
✓ Delivery parser
✓ Safe file processing
✓ Match Table
✓ Delivery Table


In [100]:
# Inspect the master tables before saving

print("=== MATCHES ===")
print("Rows:", len(matches_df))
print("Columns:", len(matches_df.columns))

print("\n=== DELIVERIES ===")
print("Rows:", len(deliveries_df))
print("Columns:", len(deliveries_df.columns))

print("\nDelivery columns:")
print(deliveries_df.columns.tolist())

=== MATCHES ===
Rows: 1
Columns: 15

=== DELIVERIES ===
Rows: 233
Columns: 23

Delivery columns:
['match_id', 'innings', 'batting_team', 'bowling_team', 'over', 'actual_delivery', 'batter', 'non_striker', 'bowler', 'runs_batter', 'runs_extras', 'runs_total', 'is_wide', 'is_no_ball', 'is_legal_delivery', 'is_dot_ball', 'is_boundary', 'is_four', 'is_six', 'is_wicket', 'player_out', 'dismissal_type', 'phase']


In [105]:
# Save processed Match Table as CSV

matches_file = processed_data_path / "matches.csv"

matches_df.to_csv(
    matches_file,
    index=False
)

print("Saved:", matches_file)

Saved: data\processed\matches.csv


In [106]:
# Save processed Delivery Table as CSV

deliveries_file = processed_data_path / "deliveries.csv"

deliveries_df.to_csv(
    deliveries_file,
    index=False
)

print("Saved:", deliveries_file)

Saved: data\processed\deliveries.csv


In [112]:
# Read the processed CSV files back into Pandas

matches_check = pd.read_csv(
    processed_data_path / "matches.csv"
)

deliveries_check = pd.read_csv(
    processed_data_path / "deliveries.csv"
)

print("Matches read back:", len(matches_check))
print("Deliveries read back:", len(deliveries_check))

Matches read back: 1
Deliveries read back: 233


In [113]:
# Compare original DataFrames with the saved CSV versions

print("Match row count preserved:",
      len(matches_check) == len(matches_df))

print("Delivery row count preserved:",
      len(deliveries_check) == len(deliveries_df))

print("\nOriginal delivery columns:")
print(deliveries_df.columns.tolist())

print("\nSaved delivery columns:")
print(deliveries_check.columns.tolist())

Match row count preserved: True
Delivery row count preserved: True

Original delivery columns:
['match_id', 'innings', 'batting_team', 'bowling_team', 'over', 'actual_delivery', 'batter', 'non_striker', 'bowler', 'runs_batter', 'runs_extras', 'runs_total', 'is_wide', 'is_no_ball', 'is_legal_delivery', 'is_dot_ball', 'is_boundary', 'is_four', 'is_six', 'is_wicket', 'player_out', 'dismissal_type', 'phase']

Saved delivery columns:
['match_id', 'innings', 'batting_team', 'bowling_team', 'over', 'actual_delivery', 'batter', 'non_striker', 'bowler', 'runs_batter', 'runs_extras', 'runs_total', 'is_wide', 'is_no_ball', 'is_legal_delivery', 'is_dot_ball', 'is_boundary', 'is_four', 'is_six', 'is_wicket', 'player_out', 'dismissal_type', 'phase']


In [114]:
# Confirm the processed files exist

for file_path in processed_data_path.iterdir():
    print(
        file_path.name,
        "|",
        round(file_path.stat().st_size / 1024, 2),
        "KB"
    )

deliveries.csv | 36.09 KB
matches.csv | 0.35 KB


In [1]:
# Final checkpoint before scaling to the full IPL dataset

print("=== PIPELINE CHECKPOINT ===")

print("JSON match files:", len(json_files))
print("Matches table:", len(matches_df))
print("Delivery table:", len(deliveries_df))

print("\nProcessed files:")
print("matches.csv:", (processed_data_path / "matches.csv").exists())
print("deliveries.csv:", (processed_data_path / "deliveries.csv").exists())

print("\nPipeline status: READY FOR FULL IPL DATA")

=== PIPELINE CHECKPOINT ===


NameError: name 'json_files' is not defined

Duplicate match rows: 0
Duplicate delivery rows: 9

PASS: No duplicate match IDs.
